### InMemoryVectorStore

In-Memory vector store implementation

uses a dictionary and computes cosine simularity for search using numpy

In [4]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

groq_llm = init_chat_model(model="groq:llama-3.3-70b-versatile")

groq_llm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EAAE667050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EAAE5DD820>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=model_name
)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10250.40it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [6]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embeddings)

In [7]:
from langchain_core.documents import Document

# Raw data records (e.g., loaded from a JSON file or API)
raw_data = [
    {
        "id": "messi_01",
        "title": "Messi's International Legacy",
        "text": "Messi has won two Copa América titles (2021, 2024) and the 2022 FIFA World Cup.",
        "player": "Lionel Messi",
        "category": "Trophies"
    },
    {
        "id": "ronaldo_01",
        "title": "Ronaldo's Champions League Record",
        "text": "Ronaldo has won 5 UEFA Champions League titles, scoring 140 goals in the competition.",
        "player": "Cristiano Ronaldo",
        "category": "Trophies"
    },
    {
        "id": "pele_01",
        "title": "Pelé's World Cup Triumph",
        "text": "Pelé remains the only player in history to win three FIFA World Cups (1958, 1962, 1970).",
        "player": "Pelé",
        "category": "World Cup"
    }
]

# Convert dictionaries into LangChain Document instances
documents = [
    Document(
        page_content=item["text"],
        metadata={
            "id": item["id"],
            "title": item["title"],
            "player": item["player"],
            "category": item["category"]
        }
    )
    for item in raw_data
]

In [9]:
print(f"Created {len(documents)} LangChain documents.")
print(documents[0])
documents

Created 3 LangChain documents.
page_content='Messi has won two Copa América titles (2021, 2024) and the 2022 FIFA World Cup.' metadata={'id': 'messi_01', 'title': "Messi's International Legacy", 'player': 'Lionel Messi', 'category': 'Trophies'}


[Document(metadata={'id': 'messi_01', 'title': "Messi's International Legacy", 'player': 'Lionel Messi', 'category': 'Trophies'}, page_content='Messi has won two Copa América titles (2021, 2024) and the 2022 FIFA World Cup.'),
 Document(metadata={'id': 'ronaldo_01', 'title': "Ronaldo's Champions League Record", 'player': 'Cristiano Ronaldo', 'category': 'Trophies'}, page_content='Ronaldo has won 5 UEFA Champions League titles, scoring 140 goals in the competition.'),
 Document(metadata={'id': 'pele_01', 'title': "Pelé's World Cup Triumph", 'player': 'Pelé', 'category': 'World Cup'}, page_content='Pelé remains the only player in history to win three FIFA World Cups (1958, 1962, 1970).')]

In [10]:
vector_store.add_documents(documents=documents)

['64633b0f-0109-426c-ab7a-46f59655d00f',
 '0aabb44c-0b12-4ac8-8247-8d3301374810',
 '71f67bcc-80fd-4aec-a79b-2da766ca00f4']

In [11]:
vector_store.similarity_search_with_score("Tell me about ronaldo trophies", k=2)

[(Document(id='0aabb44c-0b12-4ac8-8247-8d3301374810', metadata={'id': 'ronaldo_01', 'title': "Ronaldo's Champions League Record", 'player': 'Cristiano Ronaldo', 'category': 'Trophies'}, page_content='Ronaldo has won 5 UEFA Champions League titles, scoring 140 goals in the competition.'),
  0.6372230151255623),
 (Document(id='64633b0f-0109-426c-ab7a-46f59655d00f', metadata={'id': 'messi_01', 'title': "Messi's International Legacy", 'player': 'Lionel Messi', 'category': 'Trophies'}, page_content='Messi has won two Copa América titles (2021, 2024) and the 2022 FIFA World Cup.'),
  0.46018068625760633)]

In [12]:
### vector store to retriever

retriever = vector_store.as_retriever()

retriever

VectorStoreRetriever(tags=['InMemoryVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x000001EABE4B8F80>, search_kwargs={})

In [14]:
# invoke

retriever.invoke("who is the goat in football?")

[Document(id='71f67bcc-80fd-4aec-a79b-2da766ca00f4', metadata={'id': 'pele_01', 'title': "Pelé's World Cup Triumph", 'player': 'Pelé', 'category': 'World Cup'}, page_content='Pelé remains the only player in history to win three FIFA World Cups (1958, 1962, 1970).'),
 Document(id='64633b0f-0109-426c-ab7a-46f59655d00f', metadata={'id': 'messi_01', 'title': "Messi's International Legacy", 'player': 'Lionel Messi', 'category': 'Trophies'}, page_content='Messi has won two Copa América titles (2021, 2024) and the 2022 FIFA World Cup.'),
 Document(id='0aabb44c-0b12-4ac8-8247-8d3301374810', metadata={'id': 'ronaldo_01', 'title': "Ronaldo's Champions League Record", 'player': 'Cristiano Ronaldo', 'category': 'Trophies'}, page_content='Ronaldo has won 5 UEFA Champions League titles, scoring 140 goals in the competition.')]

# 🧠 Understanding `InMemoryVectorStore` in LangChain

An **`InMemoryVectorStore`** is an ephemeral, zero-setup vector database class provided directly by LangChain (`langchain_core.vectorstores`). It stores all document texts, metadata, and numerical vector embeddings in system **RAM (Random Access Memory)** rather than saving them to a local disk or an external database server.

---

## 🔑 Key Features

1. **Transient (Non-Persistent):** Data lives only while the Python process or Jupyter kernel is running. Once you restart the kernel or terminate the script, all stored vectors are deleted.
2. **Zero Configuration:** Requires no external database servers, API keys, network configuration, or disk storage paths.
3. **High Speed (Low Latency):** Because vector distance calculations happen directly in RAM, similarity lookups are practically instantaneous for small to medium datasets.
4. **Standard Runnable Interface:** Implements all standard vector store methods (`add_documents`, `similarity_search`, `as_retriever`), making it interchangeable with persistent vector stores like FAISS, Chroma, or Pinecone in LCEL chains.

---

## 📊 When to Use `InMemoryVectorStore` vs. Alternatives

| Feature | `InMemoryVectorStore` | Local File-Based (FAISS / Chroma) | Managed Cloud DB (Pinecone / Qdrant) |
| :--- | :--- | :--- | :--- |
| **Storage Medium** | System RAM | Local Disk (`.faiss`, SQLite, `.pkl`) | Remote Server / Cloud Cluster |
| **Data Persistence** | ❌ Lost when session ends | ✅ Saved locally across restarts | ✅ Saved permanently in the cloud |
| **Setup Complexity** | None (Built-in) | Low (Local directory configuration) | Medium (API keys & connection strings) |
| **Best Used For** | Prototyping, unit tests, ephemeral UI sessions | Local development, desktop RAG tools | Large-scale, multi-user production applications |

---

## 🛠️ Common Use Cases

* **Interactive Prototyping:** Experimenting with text chunking, prompt templates, or LCEL pipelines without cluttering your file system.
* **Unit & Integration Testing:** Running automated tests in CI/CD pipelines where you want a clean slate after every test run.
* **Single-Session Web Apps:** Building "upload a document and ask questions" tools (e.g., Streamlit/FastAPI) where data should be discarded after the user leaves the session.